# PZ data challenge: task set 1 end to end

Runs `cnnpz` on one task set 1 configuration, from the challenge HDF5 files to a validated subtask 1 file:

1. Read the challenge catalogues with `cnnpz.read_catalog` and build CNN inputs with `cnnpz.catalog_to_XY`.
2. Train the CNN ensemble.
3. Measure performance with the metrics the challenge scores.
4. Write the test-set p(z) in qp format and run the challenge's own format check.

**The test files have no redshifts**, so performance is measured on a held-out fraction of the training set (`VAL_FRAC`). In task set 1 the training and test sets are drawn from the same distribution, so this is a fair proxy.

**Metrics match the challenge's definitions.** The point metrics (bias, scatter, outlier rate) use the same biweight statistics as RAIL's `PZPlotterPointEstimateVsTrueHist2D`. The PIT metrics use `qp.metrics.pit.PIT`, which is what RAIL's `PZPlotterPITQQ` calls.

**The p(z) is not yet calibrated.** Each object's Gaussian width is the spread of the ensemble members, so expect poor PIT scores until item 2 of `SUBMISSION_PLAN.md` is done.

Run this from the repo root with the `qp` conda env as the kernel. The filter curves are the ones shipped with RAIL, which `pz_data_challenge` installs.

## Configuration

In [ ]:
from rail.utils.path_utils import find_rail_file

SIM = "cardinal"  # "cardinal" or "flagship"
SCENARIO = "1yr"  # "1yr" or "10yr"

DATA_DIR = "pz_data_challenge/tests/public"
# Filter curves shipped with RAIL; the challenge's catalogs.yaml ("cardinal_roman_rubin") maps the
# catalogue columns to these DC2LSST_* and roman_* filters
FILTER_DIR = find_rail_file("examples_data/estimation_data/data/FILTER")
OUT_DIR = "outputs/taskset_1"

# band -> (catalogue magnitude column, filter curve file), ordered by wavelength
BANDS = {
    "u": ("mag_u_lsst", f"{FILTER_DIR}/DC2LSST_u.res"),
    "g": ("mag_g_lsst", f"{FILTER_DIR}/DC2LSST_g.res"),
    "r": ("mag_r_lsst", f"{FILTER_DIR}/DC2LSST_r.res"),
    "i": ("mag_i_lsst", f"{FILTER_DIR}/DC2LSST_i.res"),
    "z": ("mag_z_lsst", f"{FILTER_DIR}/DC2LSST_z.res"),
    "y": ("mag_y_lsst", f"{FILTER_DIR}/DC2LSST_y.res"),
    "Y": ("mag_Y_roman", f"{FILTER_DIR}/roman_Y106.res"),
    "J": ("mag_J_roman", f"{FILTER_DIR}/roman_J129.res"),
    "H": ("mag_H_roman", f"{FILTER_DIR}/roman_H158.res"),
}
REF_BAND = "i"  # magnitudes are normalized by this band

VAL_FRAC = 0.2  # fraction of the training set held out for evaluation
N_SPLITS = 5  # ensemble members (K-fold); 10 in the full setup
EPOCHS = 100  # max epochs per member (early stopping usually ends sooner)
SEED = 42

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from qp.metrics.pit import PIT
from sklearn.model_selection import train_test_split

from pz_data_challenge import scoring, submit_utils

import cnnpz  # run from the repo root so this picks up the local source

os.makedirs(OUT_DIR, exist_ok=True)
print("cnnpz from", cnnpz.__file__)

## Load data and build features

In [ ]:
train_file = os.path.join(DATA_DIR, f"pz_challenge_taskset_1_{SIM}_training_{SCENARIO}.hdf5")
test_file = os.path.join(DATA_DIR, f"pz_challenge_taskset_1_{SIM}_test_{SCENARIO}.hdf5")

train = cnnpz.read_catalog(train_file)
test = cnnpz.read_catalog(test_file)
print(f"training: {len(train)} objects, test: {len(test)} objects")

band_columns = [column for column, _ in BANDS.values()]
print("non-detection fraction per band (training):")
print(train[band_columns].isna().mean().round(4).to_string())

In [ ]:
X, Y = cnnpz.catalog_to_XY(train, BANDS, REF_BAND)
X_test, _ = cnnpz.catalog_to_XY(test, BANDS, REF_BAND)
print("X:", X.shape, "X_test:", X_test.shape)

cnnpz.visualize_the_data(X, Y, np.arange(X.shape[1]), title=f"{SIM} {SCENARIO} training examples")
plt.xlabel("wavelength bin")
plt.show()

## Hold out a validation set and train

In [ ]:
idx_fit, idx_val = train_test_split(np.arange(len(Y)), test_size=VAL_FRAC, random_state=SEED)
X_fit, Y_fit = X[idx_fit], Y[idx_fit]
X_val, Y_val = X[idx_val], Y[idx_val]
val = train.iloc[idx_val].reset_index(drop=True)
print(f"fit: {len(Y_fit)}, validation: {len(Y_val)}")

In [ ]:
trained_models, histories = cnnpz.train_ensembles(
    cnnpz.build_model, X_fit, Y_fit, N_SPLITS=N_SPLITS, EPOCHS=EPOCHS, random_state=SEED
)

In [ ]:
cnnpz.plot_ensemble_losses(histories, ylim=(0, 0.1))

## Performance on the held-out validation set

In [ ]:
pz_val = cnnpz.ensemble_predict(trained_models, X_val, ids=val["object_id"].to_numpy())
z_point = np.squeeze(pz_val.ancil["zmode"])

### Point estimates

Δz = (z_phot − z_true) / (1 + z_true). The bias, scatter and |Δz| > 0.2 outlier rate are the scored point metrics.

In [ ]:
redshift_bins = np.linspace(0, 2.5, 11)
imag_bins = np.linspace(18, 25.5, 11)
mag_i_val = val["mag_i_lsst"].to_numpy()
point_stats, redshift_stats, imag_stats = cnnpz.get_all_stats(
    Y_val, z_point, mag_i_val, save=False, redshift_bins=redshift_bins, imag_bins=imag_bins
)
cnnpz.plot_stats(point_stats, redshift_stats, imag_stats, Y_val, z_point, redshift_bins, imag_bins, mag_i_val)
plt.show()

mean, mean_err, std, outlier_rate, abs_outlier_rate = point_stats
print(f"bias = {mean:.4f} ± {mean_err:.4f}, scatter = {std:.4f}, |dz|>0.2 outlier rate = {abs_outlier_rate:.4f}")

### p(z) calibration (PIT)

The PIT is the CDF of each object's p(z) evaluated at its true redshift. For a well-calibrated p(z) it is uniform. A U shape means the p(z) is too narrow; a hump means it is too wide.

In [ ]:
pit = PIT(pz_val, Y_val)
pit_values = pit.pit_samps

pit_stats = {
    "ks": pit.evaluate_PIT_KS().statistic,
    "CvM": pit.evaluate_PIT_CvM().statistic,
    "ksamp": pit.evaluate_PIT_anderson_ksamp().statistic,
    "outlier": pit.evaluate_PIT_outlier_rate(),
}

fig, axarr = plt.subplots(1, 2, figsize=[11, 4])
axarr[0].hist(pit_values, bins=50, range=(0, 1), density=True, histtype="step", color="k")
axarr[0].axhline(1, color="r", ls="--")
axarr[0].set_xlabel("PIT")
axarr[0].set_ylabel("density")

q = np.linspace(0, 1, 101)
axarr[1].plot(q, np.quantile(pit_values, q), "k")
axarr[1].plot([0, 1], [0, 1], "r--")
axarr[1].set_xlabel("uniform quantile")
axarr[1].set_ylabel("PIT quantile")
plt.tight_layout()
plt.show()

print({k: round(float(v), 4) for k, v in pit_stats.items()})

# qp derives "outlier" from a spline fit to the PIT quantiles, which breaks down (huge or negative
# values) when many PIT values sit at exactly 0 or 1, i.e. when p(z) is far too narrow. The challenge
# evaluator uses the same qp call; this is the direct fraction it is meant to approximate.
pit_outlier_direct = np.mean((pit_values < 0.0001) | (pit_values > 0.9999))
print(f"direct PIT outlier fraction (PIT < 0.0001 or > 0.9999): {pit_outlier_direct:.4f}")

### Against the challenge scoring tiers

`pz_data_challenge.scoring.metric_dict` defines three nested ranges per metric (tight, medium, loose), and a metric earns one point per range it falls inside, so 0–3 points each.

In [ ]:
metric_values = {"mean": mean, "std": std, "abs_outlier_rate": abs_outlier_rate, **pit_stats}

print(f"{'metric':18s} {'value':>10s}  points")
total = 0
for name, ranges in scoring.metric_dict.items():
    value = float(metric_values[name])
    points = sum(lo <= value <= hi for lo, hi in ranges)
    total += points
    print(f"{name:18s} {value:10.4f}  {points}/{len(ranges)}")
n_max = sum(len(r) for r in scoring.metric_dict.values())
print(f"{'total':18s} {'':10s}  {total}/{n_max}")

## Subtask 1: p(z) for the test set

Writes the file in the name and format the challenge expects, then runs the challenge's own check. It returns the list of checks passed, which must include all seven:

1. the file exists;
2. it contains a valid qp ensemble;
3. the ensemble has ancillary data;
4. the ancillary data is not empty;
5. it includes `zmode`;
6. it includes `object_id`;
7. the `object_id`s match the test file.

The model here was trained on the fit split only. For the real submission, train on the full training set.

In [ ]:
pz_test = cnnpz.ensemble_predict(trained_models, X_test, ids=test["object_id"].to_numpy())

estimate_file = os.path.join(OUT_DIR, f"pz_challenge_taskset_1_{SIM}_pz_estimate_{SCENARIO}.hdf5")
pz_test.write_to(estimate_file)

checks = submit_utils.check_pz_submission_file(estimate_file, test_file)
print(estimate_file, "checks passed:", checks)
assert checks == [1, 2, 3, 4, 5, 6, 7], "submission file failed the challenge format checks"